<a href="https://colab.research.google.com/github/Gabo190594/recommender-system-supermarket/blob/main/notebooks/04_reward_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Clonar tu repositorio desde GitHub
!git clone https://github.com/Gabo190594/recommender-system-supermarket.git

# Entrar a la carpeta del proyecto
%cd recommender-system-supermarket

Cloning into 'recommender-system-supermarket'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 69 (delta 25), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 586.13 KiB | 4.80 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/recommender-system-supermarket/recommender-system-supermarket/recommender-system-supermarket


In [9]:
from IPython.display import Markdown, display

display(Markdown("""
# 04. Sistema de recompensa

## Objetivo
Construir un criterio cuantitativo que permita medir el nivel de afinidad entre un usuario y un producto.

## ¿Por qué se necesita este paso?
El dataset contiene dos tipos de señales de preferencia:

- **Señal explícita:** el rating que un usuario asigna a un producto.
- **Señal implícita:** el comportamiento observado del usuario, representado por el tipo de interacción y el `implicit_score`.

Ambas señales aportan información complementaria. El rating expresa una opinión directa, mientras que la interacción refleja interés observado en el comportamiento real.

## Criterio utilizado
El criterio consiste en transformar ambas señales a una misma escala y luego combinarlas en una sola métrica:

1. El rating se normaliza entre 0 y 1.
2. El `implicit_score` normalizado se utiliza como medida de interés implícito.
3. Ambas señales se combinan mediante una ponderación.

La fórmula general es:

`reward_final = 0.6 * reward_rating + 0.4 * reward_implicit`

## Interpretación
Un valor alto de `reward_final` indica una mayor afinidad entre el usuario y el producto.
Esta variable no representa una recomendación final, sino una medida de utilidad que será utilizada en la siguiente etapa para construir el sistema de recomendación.

## Resultado esperado
Obtener un dataset enriquecido con la columna `reward_final`, que servirá como base para modelar posteriormente la lógica de recomendación.
"""))


# 04. Sistema de recompensa

## Objetivo
Construir un criterio cuantitativo que permita medir el nivel de afinidad entre un usuario y un producto.

## ¿Por qué se necesita este paso?
El dataset contiene dos tipos de señales de preferencia:

- **Señal explícita:** el rating que un usuario asigna a un producto.
- **Señal implícita:** el comportamiento observado del usuario, representado por el tipo de interacción y el `implicit_score`.

Ambas señales aportan información complementaria. El rating expresa una opinión directa, mientras que la interacción refleja interés observado en el comportamiento real.

## Criterio utilizado
El criterio consiste en transformar ambas señales a una misma escala y luego combinarlas en una sola métrica:

1. El rating se normaliza entre 0 y 1.
2. El `implicit_score` normalizado se utiliza como medida de interés implícito.
3. Ambas señales se combinan mediante una ponderación.

La fórmula general es:

`reward_final = 0.6 * reward_rating + 0.4 * reward_implicit`

## Interpretación
Un valor alto de `reward_final` indica una mayor afinidad entre el usuario y el producto.
Esta variable no representa una recomendación final, sino una medida de utilidad que será utilizada en la siguiente etapa para construir el sistema de recomendación.

## Resultado esperado
Obtener un dataset enriquecido con la columna `reward_final`, que servirá como base para modelar posteriormente la lógica de recomendación.


In [10]:
import pandas as pd
import numpy as np
from IPython.display import Markdown, display

# =========================
# CARGA
# =========================
df = pd.read_csv("data/processed/prepared_data.csv")

display(df.head())

# =========================
# REWARD EXPLÍCITO
# =========================
def reward_from_rating(rating):
    return (rating - 1) / 4

df["reward_rating"] = df["rating"].apply(reward_from_rating)

# =========================
# REWARD IMPLÍCITO
# =========================
# Ya tienes implicit_score_norm desde la preparación
df["reward_implicit"] = df["implicit_score_norm"]

# =========================
# REWARD FINAL
# =========================
def combined_reward(rating_reward, implicit_reward, w_rating=0.6, w_implicit=0.4):
    return w_rating * rating_reward + w_implicit * implicit_reward

df["reward_final"] = df.apply(
    lambda row: combined_reward(
        row["reward_rating"],
        row["reward_implicit"],
        w_rating=0.6,
        w_implicit=0.4
    ),
    axis=1
)

# =========================
# RESUMEN
# =========================
display(Markdown(f"""
## 🎯 Reward system generado

- Reward promedio: **{df['reward_final'].mean():.4f}**
- Reward mínimo: **{df['reward_final'].min():.4f}**
- Reward máximo: **{df['reward_final'].max():.4f}**
"""))

display(df[[
    "user_id", "product_id", "rating", "implicit_score",
    "reward_rating", "reward_implicit", "reward_final"
]].head())

# =========================
# GUARDAR
# =========================
df.to_csv("data/processed/reward_data.csv", index=False)
print("Archivo guardado en: data/processed/reward_data.csv")

,user_id,product_id,event_type,timestamp,implicit_score,category,price,age,segment,rating,month,dayofweek,age_norm,price_norm,implicit_score_norm,rating_norm
0,320,343,view,2025-05-12,1,lacteos,49.18,44,familia,3.0,5,0,0.565217,0.484408,0.0,0.5
1,138,367,view,2025-03-04,1,snacks,36.56,31,familia,3.0,3,1,0.282609,0.356221,0.0,0.5
2,38,465,purchase,2025-05-12,5,frutas,80.98,54,soltero,3.0,5,0,0.782609,0.807415,1.0,0.5
3,111,241,view,2025-05-13,1,bebidas,12.61,56,familia,3.0,5,1,0.826087,0.112951,0.0,0.5
4,480,199,view,2025-05-18,1,bebidas,43.19,51,soltero,3.0,5,6,0.717391,0.423565,0.0,0.5



## 🎯 Reward system generado

- Reward promedio: **0.4002**
- Reward mínimo: **0.0000**
- Reward máximo: **1.0000**


,user_id,product_id,rating,implicit_score,reward_rating,reward_implicit,reward_final
0,320,343,3.0,1,0.5,0.0,0.3
1,138,367,3.0,1,0.5,0.0,0.3
2,38,465,3.0,5,0.5,1.0,0.7
3,111,241,3.0,1,0.5,0.0,0.3
4,480,199,3.0,1,0.5,0.0,0.3


Archivo guardado en: data/processed/reward_data.csv
